In [94]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field
from uuid import uuid4

from pydantic import BaseModel, Field

from langgraph.prebuilt import create_react_agent
from langchain_core.tools import StructuredTool
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

In [64]:
load_dotenv()

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")

In [95]:
@dataclass
class OrderItem:
    name: str
    price: float
    description: str


@dataclass
class OrderData:
    id: str
    name: str
    items: List[OrderItem] = field(default_factory=list)
    status: str = 'pending'


orders: List[OrderData] = [OrderData("0", "candies"), OrderData("1", "test")]

In [ ]:
def create_order(name: str):
    """
    Description: Создание заказа

    Args:
        name (str): Название заказа.

    Returns:
        dict: Статус заказа.
    """
    order = OrderData(name=name, items=[], id=str(uuid4()))
    orders.append(order)
    return {"status": "success"}


def add_item_to_order(order_idx: int, item_name: str, price: float, description: str):
    """
    Description: Добавление товара к заказу

    Args:
        order_idx (int): номер заказа.
        item_name (str): название товара.
        price (float): цена товара.
        description (str): описание товара.

    Returns:
        dict: Статус добавления товара к заказу.
    """
    item = OrderItem(name=item_name, price=price, description=description)
    orders[order_idx].items.append(item)
    return {"status": "success"}


def remove_item_from_order(order_idx: int, item_idx: int):
    """
    Description: Удаление товара из заказа

    Args:
        order_idx (int): номер заказа.
        item_idx (int): номер товара.

    Returns:
        dict: Статус удаления товара из заказа.
    """
    orders[order_idx].items.pop(item_idx)
    return {"status": "success"}


def get_order_items(order_idx: int):
    """
    Description: Получить товары заказа

    Args:
        order_idx (int): номер заказа.

    Returns:
        list: товары заказа
    """

    return orders[order_idx].items


def get_orders():
    """
    Description: Получить заказы

    Returns:
        list: заказы
    """

    return orders

In [67]:
from langchain_gigachat import GigaChat

llm = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
)

In [97]:
tools = [create_order, get_orders, add_item_to_order, remove_item_from_order, get_order_items] 

# tools = [
#     StructuredTool.from_function(
#         func=create_order,
#         name="create_order",
#         description="Создать заказ. Аргументы: name (str)",
#         args_schema=BaseModel  # ПУСТАЯ схема!
#     ),
#     StructuredTool.from_function(
#         func=lambda order_idx, item_name, price, description: add_item_to_order(order_idx, item_name, price, description),
#         name="add_item_to_order", 
#         description="Добавить товар в заказ",
#         args_schema=BaseModel
#     )
# ]

# model = llm.bind_tools(tools)
# response = model.invoke("Создай заказ 'Тестовый'")

system_prompt = "Ты бот-продавец."

agent = create_react_agent(llm, tools, prompt=system_prompt)

C:\Users\Tumbi\AppData\Local\Temp\ipykernel_22236\318456543.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


In [69]:
result = agent.invoke({
    "messages": [HumanMessage(content="Выведи все заказы")]
})
print(result["messages"][-1].content)

Найден 1 заказ:

ID: 1
Название: test
Статус: pending

Хотите разместить новый заказ?


In [112]:
result = agent.invoke({
    "messages": [HumanMessage(content="создай заказ с названием: Тест")]
})
print(result["messages"][-1].content)

Заказ с названием «Тест» создан. Теперь Вы можете начать добавлять товары в этот заказ.


In [71]:
orders

[OrderData(id='1', name='test', items=None, status='pending'),
 OrderData(id='2fe86a77-a6e2-4964-9ffd-365178dbb213', name='Тест', items=[], status='pending')]

In [81]:
result = agent.invoke({
    "messages": [HumanMessage(content="добавь товар 'новогодняя игрушка' к заказу под номером 1 с ценой 100 рублей")]
})
print(result["messages"][-1].content)

Товар "новогодняя игрушка" успешно добавлен к заказу под номером 1. Цена товара составляет 100 рублей.


In [82]:
orders

[OrderData(id='1', name='test', items=None, status='pending'),
 OrderData(id='2fe86a77-a6e2-4964-9ffd-365178dbb213', name='Тест', items=[OrderItem(name='новогодняя игрушка', price=100.0, description='новогодняя игрушка')], status='pending')]

In [115]:
result = agent.invoke({
    "messages": [HumanMessage(content="удали товар 'новогодняя игрушка' из заказа под номером 1")]
})
print(result["messages"][-1].content)

TypeError: list indices must be integers or slices, not str

In [79]:
orders

[OrderData(id='1', name='test', items=None, status='pending'),
 OrderData(id='2fe86a77-a6e2-4964-9ffd-365178dbb213', name='Тест', items=[], status='pending')]

In [98]:
result = agent.invoke({
    "messages": [HumanMessage(content="добавь товар 'подарок другу' к заказу под номером 1 с ценой 1000 рублей")]
})
print(result["messages"][-1].content)

result = agent.invoke({
    "messages": [HumanMessage(content="добавь товар 'газировка' к заказу под номером 1 с ценой 80 рублей")]
})
print(result["messages"][-1].content)

result = agent.invoke({
    "messages": [HumanMessage(content="добавь товар 'конфета' к заказу под номером 0 с ценой 120 рублей")]
})
print(result["messages"][-1].content)

Товар "подарок другу" успешно добавлен к заказу под номером 1. Цена товара составляет 1000 рублей.
Товар "газировка" успешно добавлен к заказу №1. Цена товара - 80 рублей.
Товар «конфета» успешно добавлен к Вашему заказу № 0 по цене 120 рублей.


In [107]:
result = agent.invoke({
    "messages": [HumanMessage(content="выведи заказы")]
})
print(result["messages"][-1].content)

Вот Ваши текущие заказы:

0: Заказ "candies"
- Конфета: 120.0

1: Заказ "test"
- Подарок другу: 1000.0
- Газировка: 80.0

Какой заказ Вас интересует?


In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="выведи товары из заказа 1")]
})
print(result["messages"][-1].content)

TypeError: list indices must be integers or slices, not str